# Spark SQL Test

This notebook tests Spark SQL functionality including:
- Temporary views and SQL queries
- Joins between DataFrames
- Parquet file I/O (tests PyArrow integration)

## 1. Create SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date
import os

# Create SparkSession with PyArrow optimization
spark = SparkSession.builder \
    .appName("Spark SQL Test") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("SparkSession created with Arrow optimization enabled!")

## 2. Generate Sample Data

In [ ]:
# Sales data
sales_data = [
    (1, "P001", "North", 1500.00, date(2024, 1, 15)),
    (2, "P002", "South", 2300.00, date(2024, 1, 16)),
    (3, "P001", "East", 1800.00, date(2024, 1, 17)),
    (4, "P003", "North", 950.00, date(2024, 1, 18)),
    (5, "P002", "West", 2100.00, date(2024, 1, 19)),
    (6, "P001", "South", 1650.00, date(2024, 1, 20)),
    (7, "P003", "East", 1100.00, date(2024, 1, 21)),
    (8, "P002", "North", 2500.00, date(2024, 1, 22)),
    (9, "P001", "West", 1400.00, date(2024, 1, 23)),
    (10, "P003", "South", 890.00, date(2024, 1, 24)),
]

sales_columns = ["sale_id", "product_id", "region", "amount", "sale_date"]
sales_df = spark.createDataFrame(sales_data, sales_columns)

# Products data
products_data = [
    ("P001", "Laptop", "Electronics"),
    ("P002", "Smartphone", "Electronics"),
    ("P003", "Headphones", "Accessories"),
]

products_columns = ["product_id", "product_name", "category"]
products_df = spark.createDataFrame(products_data, products_columns)

print("Sales DataFrame:")
sales_df.show()

print("Products DataFrame:")
products_df.show()

## 3. Create Temporary Views for SQL Queries

In [ ]:
# Register DataFrames as temporary views
sales_df.createOrReplaceTempView("sales")
products_df.createOrReplaceTempView("products")

print("Temporary views created: 'sales', 'products'")

## 4. SQL Queries

In [ ]:
# Basic SELECT with WHERE
print("Sales greater than $1500:")
spark.sql("""
    SELECT sale_id, product_id, amount, region
    FROM sales
    WHERE amount > 1500
    ORDER BY amount DESC
""").show()

In [ ]:
# GROUP BY with aggregations
print("Sales by Region:")
spark.sql("""
    SELECT 
        region,
        COUNT(*) as num_sales,
        ROUND(SUM(amount), 2) as total_sales,
        ROUND(AVG(amount), 2) as avg_sale
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
""").show()

In [ ]:
# Sales by Product
print("Sales by Product:")
spark.sql("""
    SELECT 
        product_id,
        COUNT(*) as num_sales,
        ROUND(SUM(amount), 2) as total_sales
    FROM sales
    GROUP BY product_id
    ORDER BY total_sales DESC
""").show()

## 5. JOIN Operations

In [ ]:
# Join sales with products
print("Sales with Product Details (SQL JOIN):")
spark.sql("""
    SELECT 
        s.sale_id,
        p.product_name,
        p.category,
        s.region,
        s.amount,
        s.sale_date
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    ORDER BY s.sale_id
""").show()

In [ ]:
# DataFrame API join (alternative to SQL)
print("Sales with Product Details (DataFrame API JOIN):")
joined_df = sales_df.join(products_df, on="product_id", how="inner")
joined_df.select("sale_id", "product_name", "category", "region", "amount").show()

In [ ]:
# Sales summary by category
print("Sales Summary by Category:")
spark.sql("""
    SELECT 
        p.category,
        COUNT(*) as num_sales,
        ROUND(SUM(s.amount), 2) as total_sales,
        ROUND(AVG(s.amount), 2) as avg_sale
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_sales DESC
""").show()

## 6. Parquet I/O Test (PyArrow Integration)

In [ ]:
# Define output path
parquet_path = "/tmp/spark_test_output"

# Write DataFrame to Parquet
print(f"Writing sales data to Parquet: {parquet_path}")
sales_df.write.mode("overwrite").parquet(parquet_path)
print("Write completed!")

In [ ]:
# Read Parquet back
print(f"Reading Parquet from: {parquet_path}")
sales_from_parquet = spark.read.parquet(parquet_path)

print(f"\nRows read: {sales_from_parquet.count()}")
print("\nData from Parquet:")
sales_from_parquet.show()

In [ ]:
# Verify schema is preserved
print("Schema from Parquet file:")
sales_from_parquet.printSchema()

## 7. Cleanup and Stop Session

In [ ]:
# Clean up temporary files
import shutil

if os.path.exists(parquet_path):
    shutil.rmtree(parquet_path)
    print(f"Cleaned up temporary directory: {parquet_path}")

# Stop SparkSession
spark.stop()
print("\nSparkSession stopped successfully!")
print("\nSpark SQL test completed!")